# Fetch Ingredients from RxNorm

## Check Python Version

In [1]:
!python --version

Python 3.11.9


## Setup Config

In [2]:
DATASET_DIR = "../../datasets/"

import pathlib
OUTPUT_DIR = pathlib.Path(DATASET_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

FILE_RX_NORM_INGREDIENTS = "rx_norm_ingredients.csv"
RX_NORM_INGREDIENTS_PATH = OUTPUT_DIR.joinpath(FILE_RX_NORM_INGREDIENTS)
print(f"All Drug RxNorm Ingredients Location: {RX_NORM_INGREDIENTS_PATH}")

FILE_RX_NORM_PRECISE_INGREDIENTS = "rx_norm_precise_ingredients.csv"
RX_NORM_PRECISE_INGREDIENTS_PATH = OUTPUT_DIR.joinpath(FILE_RX_NORM_PRECISE_INGREDIENTS)
print(f"All Drug RxNorm Precise Ingredients Location: {RX_NORM_PRECISE_INGREDIENTS_PATH}")

FILE_UNRESOLVED_RX_NORM_PRECISE_INGREDIENTS = "unresolved_rx_norm_precise_ingredients.csv"
UNRESOLVED_RX_NORM_PRECISE_INGREDIENTS_PATH = OUTPUT_DIR.joinpath(FILE_UNRESOLVED_RX_NORM_PRECISE_INGREDIENTS)
print(f"Unresolved Drug RxNorm Precise Ingredients Location: {UNRESOLVED_RX_NORM_PRECISE_INGREDIENTS_PATH}")

Dataset Directory: ..\..\datasets
All Drug RxNorm Ingredients Location: ..\..\datasets\rx_norm_ingredients.csv
All Drug RxNorm Precise Ingredients Location: ..\..\datasets\rx_norm_precise_ingredients.csv
Unresolved Drug RxNorm Precise Ingredients Location: ..\..\datasets\unresolved_rx_norm_precise_ingredients.csv


## Create RxNormAllRelatedResolver

In [3]:
import time
import requests

class RxNormAllRelatedResolver:
    BASE_URL = "https://rxnav.nlm.nih.gov/REST"

    def get_related(self, rx_cui: str, tty: str):
        try:
            url = f"{self.BASE_URL}/rxcui/{rx_cui}/allrelated.json"
    
            response = requests.get(
                url,
                timeout=20
            )
    
            response.raise_for_status()
            data = response.json()
    
            concepts = (
                data.get("allRelatedGroup", {})
                .get("conceptGroup", [])
            )
    
            if not concepts:
                return None

            results = []
            for item in concepts:
                if item.get("tty") != tty:
                    continue

                properties = item.get("conceptProperties", [])
                for item_property in properties:
                    related_rx_cui = item_property.get("rxcui")
                    related_ingredient_name = item_property.get("name")
                    results.append({
                        "rx_cui": related_rx_cui,
                        "name": related_ingredient_name
                    })
                
            return results
        except Exception as e:
            print(f"[ALLRELATED ERROR] {tty}: {e}")
            return None
            
    def get_precise_ingredients(self, rxcui):
        precise_ingredients = self.get_related(rxcui, "PIN")
        if not precise_ingredients:
            return {
                "status": "unknown_precise_ingredients",
                "data": {
                    "rxcui": rxcui
                }
            }
            
        return {
            "status": "ok",
            "data": {
                "rxcui": rxcui,
                "precise_ingredients": precise_ingredients
            }
        }

In [4]:
resolver = RxNormAllRelatedResolver()
print(resolver.get_precise_ingredients(618442))

{'status': 'ok', 'data': {'rxcui': 618442, 'precise_ingredients': [{'rx_cui': '38253', 'name': 'tiaprofenic acid'}]}}


In [5]:
import requests
import pandas as pd
from tqdm import tqdm

df = pd.read_csv(RX_NORM_INGREDIENTS_PATH)
print(f"Total Ingredients need to be resolved for Precise Ingredients: {df.shape[0]}")

Total Ingredients need to be resolved for Precise Ingredients: 14663


## Fetch All Ingredients in RxNorm

In [6]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


results = []
unknown_pin = []

# Maksimal 5 request diproses secara paralel
resolver = RxNormAllRelatedResolver()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            resolver.get_precise_ingredients,
            row["rx_cui"],
        ): row
        for idx, row in df.iterrows()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):
        try:
            result = future.result()

            if result["status"] == "ok":
                results.append(result["data"])

            elif result["status"] == "unknown_precise_ingredients":
                unknown_pin.append(result["data"])

        except Exception as e:
            print(f"[ERROR] {futures[future]}: {e}")

100%|██████████| 14663/14663 [1:37:46<00:00,  2.50it/s]  


In [7]:
df_result = pd.DataFrame(results)
df_result.to_csv(RX_NORM_PRECISE_INGREDIENTS_PATH, index=False)

df_unknown = pd.DataFrame(unknown_pin)
df_unknown.to_csv(UNRESOLVED_RX_NORM_PRECISE_INGREDIENTS_PATH, index=False)